# Session 34 · Multimodal RAG — Practical**Phase 9 — RAG Systems.** Companion to the Session 34b slides.Duration: ~30 minutes live. CLIP and FAISS run locally on CPU. The only paid calls are thetext embeddings (fractions of a cent) and generation (a few cents).---## What we are buildingA RAG pipeline whose index contains **figures as well as text**. By the end of the notebookyou will be able to ask *"which figure shows the encoder-decoder stack?"* and get back theactual diagram from *Attention Is All You Need*, plus a grounded answer that reads it.| Step | What happens | Minutes ||---|---|---|| 0 | Setup, corpus, API check | 3 || 1 | Extract figures **and** text from real PDFs | 5 || 2 | Two encoders, two jobs — CLIP image, CLIP text, and a text-embedding model | 4 || 3 | Two FAISS legs and one metadata store | 3 || 4 | Cross-modal retrieval: a text query that finds a picture | 5 || 5 | Measuring the modality gap on our own data | 3 || 6 | Fusing the two legs with RRF | 3 || 7 | Grounded generation with a vision model | 5 || 8 | Three drills that show what breaks | 4 || 9 | Recap, knobs, exercises | — |## What you already have from Sessions 32–34aChunking, FAISS, dense retrieval, BM25 + RRF, cross-encoder reranking, grounded prompting,and a hit-rate / MRR harness. **Almost none of that changes today.** The only new ideas are:1. an encoder that turns an *image* into a vector in the same space as text, and2. the honest handling of the fact that the space is not as shared as it looks.

---## Step 0 · Setup and the corpus### WhatInstall the stack, choose three papers, and confirm the API key works before anythingexpensive runs.### Why these three papers*Attention Is All You Need*, *An Image is Worth 16x16 Words* (ViT) and *LoRA* — Sessions 23,16 and 28. You already know what is in them, which means **you can check the retrievalyourself**. That matters more than corpus size: a demo you cannot falsify teaches nothing.They are also single-column PDFs. Figure extraction on two-column papers is materiallyharder, and we come back to that in Step 1.

In [ ]:
%pip install -q pymupdf transformers torch pillow faiss-cpu openai numpy matplotlib

In [ ]:
import os, re, io, json, time, base64, urllib.requestfrom pathlib import Pathimport numpy as npimport fitz                      # PyMuPDFfrom PIL import ImagePAPERS = {    "attention": ("1706.03762", "Vaswani et al. (2017), Attention Is All You Need"),    "vit":       ("2010.11929", "Dosovitskiy et al. (2021), An Image is Worth 16x16 Words"),    "lora":      ("2106.09685", "Hu et al. (2022), LoRA: Low-Rank Adaptation of LLMs"),}CLIP_MODEL = "openai/clip-vit-base-patch32"   # 151M params, 512-d, CPU-friendlyTEXT_EMBED = "text-embedding-3-small"         # same as Sessions 32b and 33CHAT_MODEL = "gpt-5.6-luna"                   # accepts image inputCORPUS = Path("mm_corpus"); CORPUS.mkdir(exist_ok=True)FIGDIR = CORPUS / "figures"; FIGDIR.mkdir(exist_ok=True)print("workspace:", CORPUS.resolve())

In [ ]:
from openai import OpenAIclient = OpenAI()      # reads OPENAI_API_KEY from the environment# One cheap call, so a missing key fails here and not twenty minutes from now.probe = client.embeddings.create(model=TEXT_EMBED, input="ping")print("text embedding dims:", len(probe.data[0].embedding))   # expect 1536

### Behind the scenes`text-embedding-3-small` returns 1,536 dimensions; CLIP ViT-B/32 returns 512. **These twospaces have nothing to do with each other** — you cannot compare a CLIP vector to an OpenAIvector, ever. That is why we build two separate FAISS indexes in Step 3 rather than one, andwhy Step 6 fuses by rank.### Question to ask before moving on> We already have a text embedding model that scores 62+ on MTEB. Why bring in CLIP, whose> text encoder is far weaker?Because the OpenAI model has never seen a pixel. CLIP is the only one of the two that can putan *image* somewhere meaningful. We use each model for the job it can actually do.

---## Step 1 · Getting figures out of PDFs### WhatDownload the three papers and pull out **two kinds of record**:* `text` records — ordinary chunks, exactly as in Session 33.* `figure` records — a cropped PNG of each figure, plus its caption and page number.### Why this step is the hard oneIn Session 33, `pypdf` gave us text and silently discarded everything else. A PDF has noconcept of "figure": it has glyphs, vector paths and embedded rasters scattered on a page.Recovering "this rectangle of the page is Figure 3" is a heuristic, and heuristics fail.Ours works like this: find a text block that starts with `Figure N:`, walk **upward** merginganything within a small vertical gap until the artwork above has been swallowed, then cropthat rectangle.

In [ ]:
for key, (arxiv_id, _) in PAPERS.items():    dest = CORPUS / f"{key}.pdf"    if dest.exists():        print(f"{key:10s} already present ({dest.stat().st_size/1e6:.1f} MB)")        continue    req = urllib.request.Request(f"https://arxiv.org/pdf/{arxiv_id}",                                 headers={"User-Agent": "Mozilla/5.0"})    dest.write_bytes(urllib.request.urlopen(req, timeout=90).read())    print(f"{key:10s} downloaded {dest.stat().st_size/1e6:.1f} MB")

In [ ]:
CAPTION = re.compile(r"^(?:Figure|Fig\.)\s*(\d+)\s*[:.]")def _items(page):    """Every layout object on the page: text blocks, vector drawings, rasters."""    out = []    for b in page.get_text("blocks"):        out.append(("text", fitz.Rect(b[:4]), " ".join(b[4].split())))    for d in page.get_drawings():        out.append(("art", d["rect"], ""))    for img in page.get_images(full=True):        try:            for r in page.get_image_rects(img[0]):                out.append(("art", r, ""))        except Exception:            pass    return outdef extract_figures(pdf_path, key, dpi=150, first_gap=40, merge_gap=14):    """One record per figure: cropped PNG + caption + page."""    doc, records = fitz.open(pdf_path), []    for pno in range(doc.page_count):        page = doc[pno]        items = _items(page)        for kind, rect, txt in items:            m = CAPTION.match(txt) if kind == "text" else None            if not m:                continue            # 1. walk upward from the caption, swallowing the artwork            top, gap = rect.y0, first_gap            for _ in range(80):                above = [q for _, q, _ in items                         if q.y1 <= top + 1 and q.y1 >= top - gap and q.y0 < top - 0.5]                if not above:                    break                nxt = min(q.y0 for q in above)                if nxt >= top - 0.5:                    break                top, gap = nxt, merge_gap                if rect.y0 - top > 0.75 * page.rect.height:                    break            if rect.y0 - top < 55:                      # too short to be a figure                continue            # 2. horizontal extent comes from the ARTWORK only. Using the text blocks            #    too would swallow a neighbouring column.            art = [q for k, q, _ in items                   if k == "art" and q.y0 >= top - 1 and q.y1 <= rect.y0 + 1                   and q.width > 20 and q.height > 20]            if not art:                                  # a table, not a figure                continue            x0 = max(page.rect.x0 + 25, min(q.x0 for q in art) - 10)            x1 = min(page.rect.x1 - 25, max(q.x1 for q in art) + 10)            pix = page.get_pixmap(dpi=dpi,                                  clip=fitz.Rect(x0, top - 3, x1, rect.y0 - 4))            # 3. reject near-blank crops            arr = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)            if (arr.mean(axis=2) < 245).mean() < 0.005:                continue            fid = f"{key}_p{pno:02d}_fig{m.group(1)}"            out = FIGDIR / f"{fid}.png"            pix.save(out)            records.append({"id": fid, "kind": "figure", "paper": key, "page": pno + 1,                            "figure": int(m.group(1)), "caption": txt, "path": str(out),                            "size": (pix.width, pix.height)})    return recordsfigures = []for key in PAPERS:    got = extract_figures(CORPUS / f"{key}.pdf", key)    figures += got    print(f"{key:10s} {len(got):2d} figures")print(f"\ntotal {len(figures)} figures")

In [ ]:
# What did we actually get? Look before you index.for r in figures[:8]:    print(f"{r['id']:22s} {r['size'][0]:4d}x{r['size'][1]:<5d} {r['caption'][:70]}")

In [ ]:
import matplotlib.pyplot as pltwanted = ["attention_p02_fig1", "vit_p02_fig1", "lora_p00_fig1"]show = [r for r in figures if r["id"] in wanted]fig, axes = plt.subplots(1, len(show), figsize=(15, 5))for ax, r in zip(np.atleast_1d(axes), show):    ax.imshow(Image.open(r["path"])); ax.axis("off")    ax.set_title(f"{r['paper']} · Figure {r['figure']}", fontsize=10)plt.tight_layout(); plt.show()

### How it works — and where it breaksRead the three crops above carefully before moving on.**The two-column problem.** LoRA's Figure 1 sits in the right-hand column of a page whoseleft column is body text. Restricting the horizontal extent to the *artwork* rectangles(step 2 in the code) is what stops the paragraph being cropped in with the diagram — deletethose two lines and you index a picture of prose.**What this heuristic still cannot do:** figures with no caption, captions placed above theartwork, figures split across a page break, subfigures with their own `(a)/(b)` captions, andanything in a two-column layout where the artwork itself spans both columns. This is exactlythe ground that Unstructured, Docling, LlamaParse and MinerU exist to cover, and in productionyou would use one of them. We hand-roll it here for the same reason we hand-rolled chunking inSession 33: so the failure modes are visible instead of hidden behind an API.### Checkpoint> The caption *"Figure 1: The Transformer — model architecture"* is text. We could have> indexed it in Session 33 without any of this. What does the cropped PNG give us that the> caption does not?Everything the caption does not say: the shape of the stack, where the residual connectionsattach, that there are two separate positional-encoding inputs. A caption is a lossy summarywritten by someone who did not know your question.

### Now the text sideThe same recursive splitter as Session 33, reproduced compactly so this notebook standsalone. We keep page numbers, because citations are the whole point.

In [ ]:
def clean(text):    text = text.replace("\ufb01", "fi").replace("\ufb02", "fl")    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)       # repair hyphenation    text = re.sub(r"\n{2,}", "\n\n", text)    text = re.sub(r"[ \t]+", " ", text)    return text.strip()def split_recursive(text, target=900, overlap=150):    """Try the most natural boundary first, fall back to cruder ones."""    seps = ["\n\n", "\n", ". ", " "]    def _split(t, depth=0):        if len(t) <= target or depth == len(seps):            return [t]        parts, buf = [], ""        for piece in t.split(seps[depth]):            cand = (buf + seps[depth] + piece) if buf else piece            if len(cand) <= target:                buf = cand            else:                if buf:                    parts.append(buf)                if len(piece) <= target:                    buf = piece                else:                    buf = ""                    parts += _split(piece, depth + 1)        if buf:            parts.append(buf)        return parts    chunks, prev = [], ""    for c in _split(text):        chunks.append((prev[-overlap:] + " " + c).strip() if prev else c)        prev = c    return [c for c in chunks if len(c) > 80]texts = []for key in PAPERS:    doc = fitz.open(CORPUS / f"{key}.pdf")    for pno in range(doc.page_count):        for i, chunk in enumerate(split_recursive(clean(doc[pno].get_text()))):            texts.append({"id": f"{key}_p{pno:02d}_c{i}", "kind": "text",                          "paper": key, "page": pno + 1, "text": chunk})print(f"{len(texts)} text chunks, {len(figures)} figures")print(f"median chunk length: {int(np.median([len(t['text']) for t in texts]))} characters")

---## Step 2 · Two encoders, two jobs### WhatLoad CLIP, and be deliberate about which encoder does which job.| Job | Model | Dim | Why ||---|---|---|---|| Embed a figure | CLIP **image** encoder | 512 | The only model here that can read pixels || Embed a query, to search figures | CLIP **text** encoder | 512 | Must land in the same space as the figures || Embed a text chunk | `text-embedding-3-small` | 1536 | Far stronger on prose, and no 77-token limit || Embed a query, to search text | `text-embedding-3-small` | 1536 | Must match the chunks |### Why not use CLIP's text encoder for everything?Because it truncates at **77 tokens**. That is not a tuning parameter — it is the size of thepositional embedding table CLIP was trained with. A 900-character chunk simply does not fit.The next cell shows the truncation happening rather than asserting it.

In [ ]:
import torchfrom transformers import CLIPModel, CLIPProcessortorch.set_grad_enabled(False)clip = CLIPModel.from_pretrained(CLIP_MODEL).eval()proc = CLIPProcessor.from_pretrained(CLIP_MODEL)print(f"{CLIP_MODEL}: {sum(p.numel() for p in clip.parameters())/1e6:.0f}M params, "      f"{clip.config.projection_dim}-d shared space")

In [ ]:
# The 77-token wall, demonstrated.long_chunk = texts[10]["text"]kept = proc.tokenizer(long_chunk, return_tensors="pt", truncation=True, max_length=77)full = proc.tokenizer(long_chunk, add_special_tokens=True)["input_ids"]print("chunk length         :", len(long_chunk), "characters")print("tokens in full chunk :", len(full))print("tokens CLIP will see :", kept["input_ids"].shape[1])print("\nwhat survives:\n", proc.tokenizer.decode(kept["input_ids"][0])[:320], "...")

### How it worksEverything after token 77 is gone. Use CLIP as your passage encoder and you are silentlyindexing the first two sentences of every chunk. This is the most common mistake in firstattempts at multimodal RAG, and it produces a system that *works* — just badly, in a way noexception will ever tell you about.So: CLIP handles the image leg and short queries. The OpenAI model handles the text leg.

In [ ]:
def embed_images(paths, batch=16):    vecs = []    for i in range(0, len(paths), batch):        imgs = [Image.open(p).convert("RGB") for p in paths[i:i + batch]]        v = clip.get_image_features(**proc(images=imgs, return_tensors="pt"))        vecs.append(torch.nn.functional.normalize(v, dim=-1).numpy())    return np.vstack(vecs).astype("float32")def embed_clip_text(strings, batch=32):    vecs = []    for i in range(0, len(strings), batch):        inp = proc(text=strings[i:i + batch], return_tensors="pt",                   padding=True, truncation=True, max_length=77)        v = clip.get_text_features(**inp)        vecs.append(torch.nn.functional.normalize(v, dim=-1).numpy())    return np.vstack(vecs).astype("float32")def embed_openai(strings, batch=96):    vecs = []    for i in range(0, len(strings), batch):        r = client.embeddings.create(model=TEXT_EMBED, input=strings[i:i + batch])        vecs += [d.embedding for d in sorted(r.data, key=lambda d: d.index)]    a = np.array(vecs, dtype="float32")    return a / np.linalg.norm(a, axis=1, keepdims=True)t0 = time.time()IMG_VECS = embed_images([f["path"] for f in figures])print(f"{len(figures)} figures -> {IMG_VECS.shape} in {time.time()-t0:.1f}s on CPU")t0 = time.time()TXT_VECS = embed_openai([t["text"] for t in texts])print(f"{len(texts)} chunks  -> {TXT_VECS.shape} in {time.time()-t0:.1f}s")

---## Step 3 · Two indexes, one metadata store### WhatOne FAISS index per modality, and a single Python list mapping each row back to its record.### Why two indexes and not oneThe mismatched dimensionality (512 vs 1536) makes one index impossible anyway. But even if thedimensions matched you would still want two: the modality gap in Step 5 means a single rankedlist is dominated by whichever modality the query came from. Two legs plus rank fusion is nota workaround — it is the correct architecture.

In [ ]:
import faissimg_index = faiss.IndexFlatIP(IMG_VECS.shape[1]); img_index.add(IMG_VECS)txt_index = faiss.IndexFlatIP(TXT_VECS.shape[1]); txt_index.add(TXT_VECS)# Same assert as Session 33: index rows and metadata rows must not drift apart.assert img_index.ntotal == len(figures)assert txt_index.ntotal == len(texts)print(f"image leg: {img_index.ntotal:4d} vectors x {IMG_VECS.shape[1]}d "      f"= {IMG_VECS.nbytes/1e3:.0f} KB")print(f"text  leg: {txt_index.ntotal:4d} vectors x {TXT_VECS.shape[1]}d "      f"= {TXT_VECS.nbytes/1e6:.1f} MB")

### Behind the scenes — the sizing arithmetic that decides your architecture`N x dims x 4 bytes`, the same rule as Session 33. Our figure leg is tiny. Scale it up:| Corpus | One vector per page | ColPali-style, ~1,000 patch vectors per page ||---|---|---|| 10,000 pages | ~60 MB | ~60 GB || 1,000,000 pages | ~6 GB | ~6 TB |That factor of a thousand is the entire practical argument between the "unified embeddings"and "page-as-image" architectures on slide 6. It is a storage decision before it is a qualitydecision.

---## Step 4 · Cross-modal retrieval### WhatA text query, embedded by CLIP's *text* encoder, searched against *image* vectors. No captionanywhere in the loop.### Why this is the moment the session turnsEverything up to here has been Session 33 with extra file handling. This cell is the newcapability: the query and the result are not the same kind of object.

In [ ]:
def search_figures(query, k=3):    D, I = img_index.search(embed_clip_text([query]), k)    return [(figures[i], float(d)) for d, i in zip(D[0], I[0])]def search_text(query, k=3):    D, I = txt_index.search(embed_openai([query]), k)    return [(texts[i], float(d)) for d, i in zip(D[0], I[0])]def show_figures(hits, title=""):    fig, axes = plt.subplots(1, len(hits), figsize=(5 * len(hits), 4.2))    for ax, (rec, score) in zip(np.atleast_1d(axes), hits):        ax.imshow(Image.open(rec["path"])); ax.axis("off")        ax.set_title(f"{rec['paper']} Fig {rec['figure']} · p{rec['page']}"                     + (f"\ncos = {score:.3f}" if score else ""), fontsize=10)    if title:        fig.suptitle(title, fontsize=12)    plt.tight_layout(); plt.show()show_figures(search_figures("the encoder-decoder architecture of the transformer"),             "query: the encoder-decoder architecture of the transformer")

In [ ]:
for q in ["a diagram showing an image split into patches",          "a heatmap of attention weights",          "a plot of accuracy against the number of trainable parameters"]:    print(f"\n{q}")    for rec, s in search_figures(q, k=3):        print(f"   {s:.3f}  {rec['paper']:9s} Fig {rec['figure']:<2d} p{rec['page']:<3d} "              f"{rec['caption'][:56]}")

### How to read those scoresTwo things will look wrong, and both are worth stopping on.**The absolute numbers are low** — typically around 0.2–0.3 for CLIP ViT-B/32, where Session33's text-to-text cosines sat at 0.3–0.5. That is not a bug and it does not mean the match isweak; it is the modality gap, which we measure directly in the next step. Whatever number yousee, **only the gaps between ranks carry information** — never the absolute value.**At least one query will retrieve something plausible but wrong.** CLIP was trained onnatural images with short web captions; scientific diagrams sit well outside thatdistribution. It reliably recognises *"this is a flowchart of boxes and arrows"* and much lessreliably *which* flowchart. That is precisely the weakness SigLIP 2 and the document-specificmodels (ColPali, Nomic Embed Multimodal) were built to fix.### Checkpoint> Every figure's caption is sitting in our metadata. Would embedding *those* with the OpenAI> model beat CLIP here?Hold the answer — Drill 1 in Step 8 measures it, and the result is not the flattering one.

---## Step 5 · Measuring the modality gap### WhatSlide 5 showed the gap as a UMAP from the paper that named it. Here we produce the number onour own figures — which is the point: this is a property you can check on any corpus in fourlines. Three averages:* image ↔ **its own** caption — should be the highest if the space were truly shared* image ↔ **another** image* caption ↔ **another** caption

In [ ]:
caps = [f["caption"] for f in figures]CAP_VECS = embed_clip_text(caps)           # CLIP text space, the same 512-d as the imagesmatched = float(np.mean(np.sum(IMG_VECS * CAP_VECS, axis=1)))def mean_offdiag(A, B):    S = (A @ B.T).astype("float64")    np.fill_diagonal(S, np.nan)    return float(np.nanmean(S))print(f"image   <-> its own caption : {matched:.3f}   <- should be the winner")print(f"image   <-> another image   : {mean_offdiag(IMG_VECS, IMG_VECS):.3f}")print(f"caption <-> another caption : {mean_offdiag(CAP_VECS, CAP_VECS):.3f}")

### How it worksRead the ordering, not the values. On CLIP you should see a matched image–caption pair score*lower* than two unrelated images do — and if your run reproduces that, stop and sit with it.The pair is semantically identical; the two images are merely both diagrams. Cosine similarityis reporting **"same modality"** far more loudly than it reports **"same meaning"**.Three consequences you can act on:1. **Never threshold a cross-modal score.** `if cos > 0.4: keep` is calibrated for text and   will discard every image you ever retrieve.2. **Never merge legs by raw score.** The text leg's 0.45 and the image leg's 0.26 are not   comparable quantities. Step 6 fuses ranks for exactly this reason.3. **Compare within a modality, not across it.** Image-to-image ranking is trustworthy;   image-versus-text ranking is not.

---## Step 6 · Fusing the legs### WhatThe same Reciprocal Rank Fusion you wrote in Session 33, now merging a text ranking and animage ranking instead of a dense ranking and a BM25 ranking.### Why the identical line of code works on a completely different problemRRF never looks at a score. `1 / (k + rank)` only needs each leg to produce an *ordering*.Whether that leg was BM25 over tokens, cosine over OpenAI vectors, or cosine over CLIP vectorsis irrelevant to it. That indifference is why it is the default in every hybrid search engine— and why the modality gap cannot hurt it.

In [ ]:
def rrf(rankings, k=60):    """rankings: lists of record ids, best first. Returns [(id, score)] sorted."""    scores = {}    for ranked in rankings:        for rank, rec_id in enumerate(ranked, start=1):            scores[rec_id] = scores.get(rec_id, 0.0) + 1.0 / (k + rank)    return sorted(scores.items(), key=lambda kv: -kv[1])BY_ID = {r["id"]: r for r in texts + figures}def multimodal_search(query, k_text=5, k_img=4, k_final=6):    t_hits = search_text(query, k_text) if k_text else []    i_hits = search_figures(query, k_img) if k_img else []    fused = rrf([[r["id"] for r, _ in t_hits], [r["id"] for r, _ in i_hits]])    return [(BY_ID[i], s) for i, s in fused[:k_final]], t_hits, i_hitsQ = "how does the vision transformer turn an image into a sequence of tokens?"fused, t_hits, i_hits = multimodal_search(Q)for rec, s in fused:    tag = "FIG " if rec["kind"] == "figure" else "TEXT"    label = rec["caption"] if rec["kind"] == "figure" else rec["text"].replace("\n", " ")    print(f"{s:.4f}  {tag} {rec['paper']:9s} p{rec['page']:<3d} {label[:62]}")

### Behind the scenes — the dominant-modality trapNotice that the fused list contains **both** kinds of record. Now set `k_text=20` and leave`k_img=4`, and re-run. Text records will crowd the top — not because they are more relevant,but because a leg contributing 20 candidates simply has more chances to score.RRF is unweighted by default, so **the pool sizes are the weights**. In production, eitherkeep the legs balanced or add an explicit per-modality weight:```pythonscores[rec_id] += weight[leg] / (k + rank)```Tune that weight on a labelled set — the same discipline as Session 33's ablation table.

---## Step 7 · Generation with a vision model### WhatSend the fused context to `gpt-5.6-luna` — text chunks as text, figures as actual images —and require a grounded answer that cites both.### Why the prompt still decidesRetrieval put a diagram in front of the model. Nothing about that forces the model to *use*it, to admit when it cannot read it, or to say which source a claim came from. As in Session33: **retrieval finds; the prompt decides.**

In [ ]:
SYSTEM = """You answer questions about machine-learning papers using ONLY the providedcontext, which contains text passages and figures.Rules:- Cite the source of every claim as [T1], [T2] for text passages and [F1], [F2] for figures.- If a claim comes from reading a figure, say so explicitly and cite that figure.- If the answer is not in the provided context, say "The provided sources do not answer  this." Do not use your own knowledge of these papers to fill the gap.- Be concise. Do not restate the question."""def to_data_url(path, max_side=768):    """Downscale before sending: images are billed per tile, not per file."""    im = Image.open(path).convert("RGB")    if max(im.size) > max_side:        r = max_side / max(im.size)        im = im.resize((int(im.width * r), int(im.height * r)))    buf = io.BytesIO(); im.save(buf, format="PNG")    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()def answer(question, k_text=4, k_img=2, show=True):    fused, _, _ = multimodal_search(question, k_text=k_text * 2,                                    k_img=k_img * 2, k_final=20)    picked_t = [r for r, _ in fused if r["kind"] == "text"][:k_text]    picked_f = [r for r, _ in fused if r["kind"] == "figure"][:k_img]    parts, legend = [], []    for n, r in enumerate(picked_t, 1):        legend.append(f"[T{n}] {r['paper']} p{r['page']}")        parts.append({"type": "input_text",                      "text": f"[T{n}] ({r['paper']}, page {r['page']})\n{r['text']}"})    for n, r in enumerate(picked_f, 1):        legend.append(f"[F{n}] {r['paper']} Fig {r['figure']} p{r['page']}")        parts.append({"type": "input_text",                      "text": f"[F{n}] ({r['paper']}, page {r['page']}) "                              f"caption: {r['caption']}"})        parts.append({"type": "input_image", "image_url": to_data_url(r["path"])})    parts.append({"type": "input_text", "text": f"\nQuestion: {question}"})    resp = client.responses.create(        model=CHAT_MODEL,        input=[{"role": "system", "content": [{"type": "input_text", "text": SYSTEM}]},               {"role": "user", "content": parts}])    if show:        if picked_f:            show_figures([(r, 0.0) for r in picked_f], "figures sent to the model")        print("context:", " | ".join(legend), "\n")        print(resp.output_text)    return resp.output_text, picked_t, picked_f_ = answer("In the Transformer architecture, what sits between the multi-head attention "           "sublayer and the feed-forward sublayer, and how do you know?")

### CheckpointLook at where the citations land. If the model answered "Add & Norm" and cited `[F1]`, thefigure did the work — that phrase appears in the diagram, and the surrounding prose describesit only indirectly. If it cited a text passage instead, the text leg happened to retrieve thesublayer description, and you have learned something about your own corpus.Either is a legitimate outcome. The failure to watch for is an answer with **no** citation, orone that cites `[F1]` for a claim the figure does not support.

---## Step 8 · Three drillsEach drill removes one thing and shows what breaks.

### Drill 1 · Would captions have been enough?The cheapest architecture on slide 6 is *caption-and-index*: never embed a pixel, just embedthe caption as text. We have the captions. Let us find out what CLIP is buying us.

In [ ]:
CAP_OPENAI = embed_openai(caps)cap_index = faiss.IndexFlatIP(CAP_OPENAI.shape[1]); cap_index.add(CAP_OPENAI)PROBES = [    ("the encoder-decoder architecture of the transformer",      "attention", 1),    ("an image split into fixed size patches",                   "vit",       1),    ("the low-rank update matrices A and B",                     "lora",      1),    ("scaled dot-product and multi-head attention side by side", "attention", 2),]def rank_of(hits, paper, fignum):    for r, (rec, _) in enumerate(hits, 1):        if rec["paper"] == paper and rec["figure"] == fignum:            return r    return Noneprint(f"{'query':<54s}{'CLIP image':>12s}{'caption text':>14s}")for q, paper, fignum in PROBES:    clip_hits = search_figures(q, k=5)    D, I = cap_index.search(embed_openai([q]), 5)    cap_hits = [(figures[i], float(d)) for d, i in zip(D[0], I[0])]    print(f"{q[:54]:<54s}{str(rank_of(clip_hits, paper, fignum)):>12s}"          f"{str(rank_of(cap_hits, paper, fignum)):>14s}")

**How to read this.** The number is the rank at which the figure we were looking for appeared;`None` means it was not in the top 5. Captions often win — a caption is human-written,on-topic and in the same language as the query, and `text-embedding-3-small` is a far strongertext model than CLIP's text tower.That is the honest result, and it is the argument for caption-and-index as a *first* move on acorpus whose figures are well captioned. CLIP earns its place where captions are absent,generic ("Figure 4: Results") or wrong — and on image-to-image search, which captions cannotdo at all. Pick the architecture from the corpus, not from the diagram on the slide.

### Drill 2 · Fusing by score instead of by rank

In [ ]:
def naive_score_fusion(query, k=6):    t = [(r["id"], s) for r, s in search_text(query, 8)]    i = [(r["id"], s) for r, s in search_figures(query, 8)]    return sorted(t + i, key=lambda kv: -kv[1])[:k]q = "how does the vision transformer turn an image into a sequence of tokens?"print("RAW SCORE FUSION")for rid, s in naive_score_fusion(q):    print(f"   {s:.3f}   {BY_ID[rid]['kind']:6s} {BY_ID[rid]['paper']}")print("\nRRF")for rec, s in multimodal_search(q)[0]:    print(f"   {s:.4f}  {rec['kind']:6s} {rec['paper']}")

Raw score fusion returns text records almost exclusively — not because the figures areirrelevant (Step 4 showed the right figure ranking first *inside its own leg*) but because0.45 beats 0.26 and the comparison is meaningless. **The modality gap does not degrade scorefusion gracefully; it deletes one modality.**

### Drill 3 · Take the figure away

In [ ]:
FIGURE_Q = ("According to the ViT model overview figure, what is prepended to the sequence "            "of patch embeddings before it enters the Transformer encoder?")print("=" * 78, "\nWITH FIGURES\n", "=" * 78)with_fig, _, _ = answer(FIGURE_Q, k_text=4, k_img=2, show=False)print(with_fig)print("\n" + "=" * 78, "\nTEXT ONLY (k_img = 0) — the Session 33 pipeline\n", "=" * 78)text_only, _, _ = answer(FIGURE_Q, k_text=4, k_img=0, show=False)print(text_only)

Compare the two. The interesting outcome is not always "text-only fails" — ViT's prose doesdescribe the `[class]` token. What changes is **which source the answer can cite**, andwhether the model can confirm what the diagram actually shows. On a corpus where the figure isthe only carrier — a wiring diagram, an unlabelled waterfall chart — the text-only answerdegrades from *citable* to *guessed*.Run it twice. Watch for the model describing the figure confidently in the text-onlycondition: that is chart hallucination arriving from training data rather than from yourindex, and it is exactly the failure a citation requirement is designed to expose.

---## Step 9 · Recap### The pipeline you just built```PDFs ──┬─→ text chunks ───→ text-embedding-3-small ──→ FAISS (1536-d) ─┐       │                                                               ├─→ RRF ─→ VLM ─→ answer       └─→ figure crops ──→ CLIP image encoder ───────→ FAISS (512-d) ─┘       + [T]/[F] cites                                                            ▲                                    query ──→ CLIP text encoder (max 77 tokens)```### The knobs, and what each one costs| Knob | Our value | What moves when you change it ||---|---|---|| Figure-extraction heuristic | caption-anchored, artwork-bounded | The recall ceiling of the entire image leg || Image encoder | CLIP ViT-B/32 | Quality on diagrams; SigLIP 2 is the obvious upgrade || Text encoder | `text-embedding-3-small` | Quality of the text leg — unchanged from Session 33 || `k_text` vs `k_img` | 8 vs 4 | The **effective weight** of each modality inside RRF || RRF `k` | 60 | How sharply the top ranks dominate || Image `max_side` | 768 px | Input tokens per figure, and whether small labels stay legible |**None of these has a correct value. Each has a measurable one** — and the Session 33 harnesstransfers unchanged: build a golden set of question → expected-figure pairs and score hit rateand MRR on the image leg exactly as you did on the text leg.### Six things to take away1. Text-only ingestion loses figures **silently**, before any chunking decision is made.2. Figure extraction is a heuristic, and yours will fail on two-column layouts first.3. CLIP's text encoder stops at 77 tokens — never use it as a passage encoder.4. The modality gap is real and measurable: a matched image–caption pair scores lower than   two unrelated images.5. Fuse ranks, never scores. Raw score fusion deletes the weaker-scoring modality.6. Captions are a strong baseline. Measure before you assume you need pixels.### Exercises1. **Golden set.** Write 10 questions whose answers live in a specific figure and record the   expected `figure_id`. Score hit-rate@3 and MRR@3 for the image leg. This is the artefact   that makes every later change safe.2. **Swap the encoder.** Replace CLIP with `google/siglip2-base-patch16-224` and re-run   exercise 1. Report the change in hit rate — not your impression of the outputs.3. **Weighted RRF.** Add a per-modality weight and sweep the image weight over   `[0.5, 1.0, 2.0]`. Which value maximises MRR on your golden set?4. **Caption enrichment.** Ask the vision model to describe each figure at index time, embed   `caption + generated description` as text, and add it as a *third* leg. Does three legs   beat two, or does it just dilute?5. **Break the extractor.** Add a two-column paper (`2103.00020`, CLIP) to `PAPERS` and look   at what comes out. Fix one failure mode.6. **Cost.** Instrument `answer()` to log input tokens. What does one figure actually cost at   `max_side` 768 versus 1024?### Where this goes next| Session | Topic | Builds on today ||---|---|---|| 35 | RAG evaluation (RAGAS) | Faithfulness and answer relevance, including figure-grounded claims || 36 | Milestone — full RAG system | Your own documents, text and figures, end to end |---*The figures used in the Session 34 slides are reproduced from the original papers withattribution; see `figures/ATTRIBUTIONS.md`. The figures extracted by this notebook come fromarXiv preprints downloaded at runtime and are used here for teaching only.*